### 消息裁剪

In [5]:
from typing import Any

from langchain_core.runnables import RunnableConfig
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model, SummarizationMiddleware
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from openai import conversations

"""
目标是控制token用量，通常保留系统初始消息喝最近若干消息，或按token
数保留末尾内容


四次对话及每次调用模型前的消息长度和修剪动作：

第一次（'你好，我是老王'）

调用前消息数：1（≤3，不修剪）

模型回复后消息数：2（Human1 + AI1）

第二次（'从现在起，你叫小王'）

调用前消息数：3（Human1, AI1, Human2，≤3，不修剪）

模型回复后消息数：4（Human1, AI1, Human2, AI2）

第三次（'今天天气不错'）

调用前消息数：5（Human1, AI1, Human2, AI2, Human3，>3）

长度奇数（5），保留 first_message（Human1） + messages[-4:]（AI1, Human2, AI2, Human3） → 共 5 条，没实质删减

模型回复后消息数：6（Human1, AI1, Human2, AI2, Human3, AI3）

第四次（'告诉我，你是谁，我是谁'）

调用前消息数：7（Human1, AI1, Human2, AI2, Human3, AI3, Human4，>3）

长度奇数（7），保留 first_message（Human1） + messages[-4:]（AI2, Human3, AI3, Human4） → 共 5 条

修剪后状态变为这 5 条，然后模型处理 Human4 并追加 AI4

最终消息列表：[Human1, AI2, Human3, AI3, Human4, AI4] → 6 条

"""
from langchain.agents import create_agent, AgentState
from llm.my_llm import model_tool

@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str,Any]|None:

    messages=state['messages']

    if len(messages) <= 3:
        return None
    first_message=messages[0]

    recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]

    new_messages=[first_message]+recent_messages

    return {
        'messages':[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent=create_agent(
    model=model_tool,
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

config:RunnableConfig={
        'configurable':{
            'thread_id':'1'
        }
    }
agent.invoke({'messages':[HumanMessage('你好，我是老王')]},config)
agent.invoke({'messages':[HumanMessage('从现在起，你叫小王')]},config)
agent.invoke({'messages':[HumanMessage('今天天气不错')]},config)

final_response=agent.invoke({'messages':[HumanMessage('告诉我，你是谁，我是谁')]},config)


for msg in final_response['messages']:
    msg.pretty_print()



================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，老王！从现在起我就是小王了。请问有什么我可以帮您的？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！好天气总是让人心情舒畅。您今天有什么安排吗？是打算出去走走晒晒太阳，还是有什么其他的计划呢？
================================ Human Message =================================

告诉我，你是谁，我是谁
================================== Ai Message ==================================

您是老王，我是小王呀！咱们刚才不是刚“认亲”了嘛。作为您的AI助手小王，我随时准备为您效劳！今天有什么需要小王帮忙的吗？


### 消息删除

In [7]:
"""
    消息裁剪：强调在模型调用前裁剪消息列表，控制模型可以看到的上下文范围
    消息删除：强调模型调用完成后将某些消息从消息列表中删除

    RemoveMessage(id='1')并不会去内存的数组把id=1的对象删掉，而是
    作为一条新纪录追加到当前线程的状态历史中
"""

from typing import Any

from langchain_core.runnables import RunnableConfig
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model,after_model
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver


from langchain.agents import create_agent, AgentState
from llm.my_llm import model_tool

@after_model
def delete_old_messages(state:AgentState,runtime:Runtime)->dict[str,Any]|None:

    messages=state['messages']

    if len(messages) > 5:
        to_delete=len(messages)-5
        return {
            'messages':[RemoveMessage(id=m.id)for m in messages[:to_delete]]
        }
    return None



agent=create_agent(
    model=model_tool,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config:RunnableConfig={
        'configurable':{
            'thread_id':'1'
        }
    }
agent.invoke({'messages':[HumanMessage('你好，我是老王')]},config)
agent.invoke({'messages':[HumanMessage('从现在起，你叫小王')]},config)
agent.invoke({'messages':[HumanMessage('今天天气不错')]},config)

final_response=agent.invoke({'messages':[HumanMessage('告诉我，你是谁，我是谁')]},config)


for msg in final_response['messages']:
    msg.pretty_print()





================================== Ai Message ==================================

没问题，老王！从现在起我就是小王了。

老王有什么吩咐，小王随时听候差遣！今天咱们聊点什么，或者有什么需要小王帮忙的吗？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！今天天气确实挺不错的，看着就让人心情舒畅。

这么好的天气，您今天有什么安排吗？是打算出门溜达溜达、晒晒太阳，还是打算在家好好歇着？要是有什么需要小王跑腿或者帮忙的，您随时吩咐啊！
================================ Human Message =================================

告诉我，你是谁，我是谁
================================== Ai Message ==================================

老王，您这是考我呢？

您是老王，我是小王呀！咱们俩这称呼可都定好了，小王随时准备为老王服务！


### 摘要

In [1]:
from langchain_core.runnables import RunnableConfig
from langgraph.graph.message import REMOVE_ALL_MESSAGES

from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from  langchain.agents.middleware import SummarizationMiddleware

from langchain.agents import create_agent, AgentState
from llm.my_llm import model_tool,model

"""
    最大token数设置原则：
        模型上下文窗口 4K->设置3000
        模型上下文窗口 8K->设置6000
        模型上下文窗口 16K->设置12000

"""

agent=create_agent(
    model=model_tool,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ('tokens',100), # 超过100 tokens就摘要
            ],
            keep=('messages',2),
            summary_prompt='对历史消息摘要，消息列表如下\n{messages}'
        )
    ]
)

config={"configurable":{
    "thread_id":'1'
}}

print("\n进行多轮对话...")

conversations=[
    "我叫张三，我是工程师。这里是一段非常长的废话"*20,
    "请总结一下我的信息"
]

for msg in conversations:
    response=agent.invoke({
        "messages":[HumanMessage(msg)]
    },config)
    for msg in response['messages']:
        msg.pretty_print()
    print("*"*50)





进行多轮对话...
================================ Human Message =================================

我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话我叫张三，我是工程师。这里是一段非常长的废话
================================== Ai Message ==================================

你好，张三工程师！👋

我已经成功接收并阅读了你这段“非常长的废话”（大概重复了20多次）。看来你是在测试我的文本处理能力，或者只是单纯想跟我开个玩笑？😄

请问今天有什么我可以帮你的吗？无论是探讨工程技术问题、写代码、做数据分析，还是真的需要我帮你生成一段“非常长的废话”，我都随时待命！
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to date:

- 用户自我介绍：**张三**，职业为**工程师**。  
- 其余内容均为重复性废话，无新增信息。
======